In [2]:
import sys
import polars as pl
import random, re
from scipy.stats import norm

In [3]:
# TODO also simulate different tails:
phosphorus = 30.973761998
oxygen = 31.989829239
d_ribose = 150.05282342
tail_5prime = [d_ribose, oxygen, 2 * oxygen + phosphorus, oxygen]

In [4]:
masses = pl.read_csv("../workflow/resources/masses_all.tsv",separator="\t")#, sep="\t").set_index("nucleoside", drop=False)
masses

nucleoside,monoisotopic_mass
str,f64
"""3483G""",508.1918
"""100G""",307.0917
"""68A""",297.1073
"""k2C""",371.180483
"""347G""",436.1706
…,…
"""2160A""",397.142
"""U""",244.06954
"""chm5U""",318.06993


In [56]:
#masses2 = pl.read("../workflow/resources/Pyramidine_modifications.xls")#,separator="\t")
masses2 = pl.read_excel("../workflow/resources/Pyramidine_modifications.xlsx")

In [57]:
masses_pyrimidines = masses2.select(
    pl.col("Symbol").alias("nucleoside"),
    pl.col("Mass").cast(pl.Float64).alias("monoisotopic_mass"),
    )

In [58]:
masses_pyrimidines

nucleoside,monoisotopic_mass
str,f64
"""C""",243.08552
"""U""",244.069536
"""A""",267.096754
"""G""",283.091668
"""C+""",352.173327
…,…
"""s4U""",260.046692
"""se2U""",307.046421
"""tm5U""",381.0842


In [59]:
masses_pyrimidines.filter(pl.col("nucleoside").is_in(["T","A","C","G","U"]))

nucleoside,monoisotopic_mass
str,f64
"""C""",243.08552
"""U""",244.069536
"""A""",267.096754
"""G""",283.091668


In [60]:
masses.filter(pl.col("nucleoside").is_in(["T","A","C","G","U"]))

nucleoside,monoisotopic_mass
str,f64
"""G""",283.09167
"""C""",243.08552
"""A""",267.09675
"""U""",244.06954


In [61]:
masses_pyrimidines.write_csv("masses_pyrimidines_modifications.tsv", separator="\t")

In [62]:
masses_all = masses.vstack(masses_pyrimidines)
masses_all

nucleoside,monoisotopic_mass
str,f64
"""3483G""",508.1918
"""100G""",307.0917
"""68A""",297.1073
"""k2C""",371.180483
"""347G""",436.1706
…,…
"""s4U""",260.046692
"""se2U""",307.046421
"""tm5U""",381.0842


In [63]:
masses_all = masses_all.unique(pl.col("nucleoside"))
masses_all

nucleoside,monoisotopic_mass
str,f64
"""02G""",311.123
"""61A""",335.1594
"""ac4C""",285.096085
"""cm5U""",302.075015
"""106G""",571.2126
…,…
"""69A""",394.1237
"""m3Y""",258.085186
"""00A""",479.1053


In [64]:
masses_all

nucleoside,monoisotopic_mass
str,f64
"""02G""",311.123
"""61A""",335.1594
"""ac4C""",285.096085
"""cm5U""",302.075015
"""106G""",571.2126
…,…
"""69A""",394.1237
"""m3Y""",258.085186
"""00A""",479.1053


In [65]:
masses_all.filter(pl.col("nucleoside").is_in(["T","A","C","G","U"]))

nucleoside,monoisotopic_mass
str,f64
"""G""",283.09167
"""U""",244.06954
"""C""",243.08552
"""A""",267.09675


In [66]:
masses_all.write_csv("masses_all.tsv", separator="\t")

In [73]:
nucleoside_re = re.compile(r"\d*[ACGU]")
true_sequence = nucleoside_re.findall("CG100GUU")
n_fragments = 10
print(true_sequence)

['C', 'G', '100G', 'U', 'U']


In [74]:
true_sequence = nucleoside_re.findall("CUA104GU1A")
print(true_sequence)
print(len(true_sequence))

['C', 'U', 'A', '104G', 'U', '1A']
6


In [75]:
# helper functions
def random_fragment():
    l = random.randint(0, len(true_sequence) - 1)
    r = random.randint(l + 1, len(true_sequence))
    return l, r

In [80]:
fragments = pl.from_records(
    [random_fragment() for _ in range(n_fragments)],
    schema=["left", "right"],
    orient="row",
)
fragments

left,right
i64,i64
0,2
1,3
3,5
3,5
4,5
1,3
4,5
2,6
3,4


In [81]:
true_fragment_masses = [
    sum(
        masses.filter(pl.col("nucleoside") == b)
        .select(pl.col("monoisotopic_mass"))
        .item()
        for b in true_sequence[fragment["left"] : fragment["right"]]
    )
    for fragment in fragments.iter_rows(named=True)
]
true_fragment_masses


[487.15506,
 511.16629,
 815.2821399999999,
 815.2821399999999,
 244.06954,
 511.16629,
 244.06954,
 1363.49129,
 571.2126,
 525.1819399999999]

In [82]:
len(true_sequence)

6

In [83]:
backbone_middle_addition = 1.
backbone_start_addition  = 2.
backbone_end_addition    = 0.

In [42]:
#Copying the fragment masses to a new list to add the backbone masses to the fragments.
true_fragment_masses_with_backbone = [elem for elem in true_fragment_masses]

# Add backbone masses to the fragments, based on the position of the fragment in the sequence!
for iter,fragment in enumerate(fragments.iter_rows(named=True)):
    
    if fragment["left"] == 0:
        true_fragment_masses_with_backbone[iter] += backbone_start_addition
    elif fragment["left"] != len(true_sequence)-1:
        true_fragment_masses_with_backbone[iter] += backbone_middle_addition
    
    if fragment["right"] == len(true_sequence)-1:
        true_fragment_masses_with_backbone[iter] += backbone_end_addition
    elif fragment["right"] != 0:
        true_fragment_masses_with_backbone[iter] += backbone_middle_addition
    
    true_fragment_masses_with_backbone[iter] += (
        backbone_middle_addition*max(fragment["right"]-fragment["left"]+1-2,0))

true_fragment_masses_with_backbone

[841.30935,
 491.15506,
 1086.37889,
 269.09675,
 1331.46441,
 817.2821399999999,
 246.08552,
 1100.39454,
 282.1124,
 491.15506]

In [85]:
#Copying the fragment masses to a new list to add the backbone masses to the fragments.
true_fragment_masses_with_backbone = [elem for elem in true_fragment_masses]
for iter,fragment in enumerate(fragments.iter_rows(named=True)):
    
        if fragment["left"] == 0:
            true_fragment_masses_with_backbone[iter] += backbone_start_addition #+ backbone_masses["Tag5prime"] #Can add a 5'tag mass if applicable here!
        else:
            true_fragment_masses_with_backbone[iter] += backbone_end_addition
        
        if fragment["right"] == len(true_sequence)-1:
            true_fragment_masses_with_backbone[iter] += 0. #+ backbone_masses["Tag3prime"] #Can add a 3'tag mass if applicable here!   
        
        true_fragment_masses_with_backbone[iter] += (
            backbone_middle_addition*max(fragment["right"]-fragment["left"],0))

true_fragment_masses_with_backbone

[491.15506,
 513.16629,
 817.2821399999999,
 817.2821399999999,
 245.06954,
 513.16629,
 245.06954,
 1367.49129,
 572.2126,
 527.1819399999999]

In [43]:
fragment_noise = norm.rvs(scale=0.0001, size=len(true_fragment_masses_with_backbone))
observed_fragment_masses = [
    max(mass + mass*noise, 0.0)
    for noise, mass in zip(fragment_noise, true_fragment_masses_with_backbone)
]

In [44]:
observed_fragment_masses

[np.float64(841.3311192864016),
 np.float64(491.1281204677023),
 np.float64(1086.4384191623762),
 np.float64(269.0965211288063),
 np.float64(1331.478372947346),
 np.float64(817.3470265948874),
 np.float64(246.08268271897614),
 np.float64(1100.502588386357),
 np.float64(282.15589716504365),
 np.float64(491.12752602348377)]

In [45]:
fragments = fragments.with_columns(
    (pl.col("left") == 0).alias("is_start"),
    (pl.col("right") == (len(true_sequence))-1).alias("is_end"),
    (pl.col("right") == pl.col("left")).alias("single_nucleotide"),
    pl.Series(true_fragment_masses).alias("true_nucleoside_mass"),
    pl.Series(true_fragment_masses_with_backbone).alias("true_mass_with_backbone"),
    pl.Series(observed_fragment_masses).alias("observed_mass"),
)
fragments

left,right,is_start,is_end,single_nucleotide,true_nucleoside_mass,true_mass_with_backbone,observed_mass
i64,i64,bool,bool,bool,f64,f64,f64
2,4,false,false,false,838.30935,841.30935,841.331119
0,2,true,false,false,487.15506,491.15506,491.12812
1,4,false,false,false,1082.37889,1086.37889,1086.438419
2,3,false,false,false,267.09675,269.09675,269.096521
0,4,true,false,false,1325.46441,1331.46441,1331.478373
3,5,false,true,false,815.28214,817.28214,817.347027
0,1,true,false,false,243.08552,246.08552,246.082683
3,6,false,false,false,1096.39454,1100.39454,1100.502588
5,6,false,false,false,281.1124,282.1124,282.155897


In [45]:
num_parts = 2
breakpoints = random.sample(range(1, len(true_sequence)), num_parts - 1)
breakpoints

[3]

In [37]:
true_sequence

['C', 'G', '100G', 'U', 'U']

In [115]:
def random_fragment(num_parts = num_parts):
    if num_parts == 1:
        breakagepoints = [int(0)]
    else:
        breakagepoints = sorted(random.sample(range(1, len(true_sequence)), num_parts - 1))
        #Implement the condition that the num_parts cannot be greater than the sequence length!
    return breakagepoints

In [116]:
[random_fragment(num_parts = random.randint(1,3)) for _ in range(n_fragments)]

[[0], [0], [0], [0], [3], [4], [2, 3], [1, 2], [0], [2, 3]]

In [84]:
[random.randint(2,2) for _ in range(n_fragments)]

[2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

In [118]:
breakagepoints = [random_fragment(num_parts = random.randint(1,3)) for _ in range(n_fragments)]
print(breakagepoints)

[[4], [0], [3], [1, 4], [1, 3], [1, 3], [4], [0], [1, 3], [1]]


In [176]:
def generate_left_right(breakagepoints):
    left,right = [],[]
    for exp in breakagepoints:
        left.append(0)
        for part in exp:
            if part != 0:
                right.append(part-1)
                left.append(part)
        right.append(len(true_sequence)-1)
    return left,right
l,r = generate_left_right(breakagepoints)
print(l)
print(r)

[0, 4, 0, 0, 3, 0, 1, 4, 0, 1, 3, 0, 1, 3, 0, 4, 0, 0, 1, 3, 0, 1]
[3, 4, 4, 2, 4, 0, 3, 4, 0, 2, 4, 0, 2, 4, 3, 4, 4, 0, 2, 4, 0, 4]


In [188]:
l_breakage,r_breakage = generate_left_right(breakagepoints) 

# sample random fragments from true sequence
fragments = pl.from_records(
    list(zip(l_breakage,r_breakage)), 
    schema=["left", "right"],
    orient="row",
)
fragments

left,right
i64,i64
0,3
4,4
0,4
0,2
3,4
…,…
0,0
1,2
3,4


In [198]:
masses = pl.read_csv("../workflow/resources/masses_all.tsv", separator="\t")#.set_index("nucleoside", drop=False)

In [199]:
masses

nucleoside,monoisotopic_mass
str,f64
"""A""",267.09675
"""G""",283.09167
"""C""",243.08552
"""U""",244.06954
"""1A""",281.1124
…,…
"""10G""",409.1597
"""102G""",425.1547
"""105G""",538.2023


In [202]:
true_fragment_masses = [
    sum(
        masses.filter(pl.col("nucleoside") == b)
        .select(pl.col("monoisotopic_mass"))
        .item()
        for b in true_sequence[fragment["left"] : fragment["right"] + 1]
    )
    for fragment in fragments.iter_rows(named=True)
]
true_fragment_masses

[1077.33843,
 244.06954,
 1321.40797,
 833.26889,
 488.13908,
 243.08552,
 834.2529099999999,
 244.06954,
 243.08552,
 590.18337,
 488.13908,
 243.08552,
 590.18337,
 488.13908,
 1077.33843,
 244.06954,
 1321.40797,
 243.08552,
 590.18337,
 488.13908,
 243.08552,
 1078.32245]

In [206]:
fragment_noise = norm.rvs(scale=0.5, size=len(true_fragment_masses))
observed_fragment_masses = [
    max(true_mass + noise, 0.0)
    for noise, true_mass in zip(fragment_noise, true_fragment_masses)
]
observed_fragment_masses

[np.float64(1077.841758723866),
 np.float64(243.862628301077),
 np.float64(1321.4848804971411),
 np.float64(833.6068379409538),
 np.float64(488.86297315968983),
 np.float64(243.28987402748135),
 np.float64(833.7682694443158),
 np.float64(244.68106449347934),
 np.float64(243.9708025231497),
 np.float64(590.0286820594827),
 np.float64(488.42035567772655),
 np.float64(242.87470715579988),
 np.float64(590.2245504801557),
 np.float64(488.6767642891667),
 np.float64(1077.6115076299168),
 np.float64(243.54411881208677),
 np.float64(1320.4671665636643),
 np.float64(243.25163640601843),
 np.float64(589.9374466155555),
 np.float64(487.4887652856656),
 np.float64(242.5098622415963),
 np.float64(1078.9379095316303)]

In [209]:
fragments = fragments.with_columns(
    (pl.col("left") == 0).alias("is_start"),
    (pl.col("right") == (len(true_sequence))-1).alias("is_end"),
    (pl.col("right") == pl.col("left")).alias("single_nucleotide"),
    pl.Series(true_fragment_masses).alias("true_mass"),
    pl.Series(observed_fragment_masses).alias("observed_mass"),
)
fragments


left,right,is_start,is_end,true_mass,observed_mass,single_nucleotide
i64,i64,bool,bool,f64,f64,bool
0,3,true,false,1077.33843,1077.841759,false
4,4,false,true,244.06954,243.862628,true
0,4,true,true,1321.40797,1321.48488,false
0,2,true,false,833.26889,833.606838,false
3,4,false,true,488.13908,488.862973,false
…,…,…,…,…,…,…
0,0,true,false,243.08552,243.251636,true
1,2,false,false,590.18337,589.937447,false
3,4,false,true,488.13908,487.488765,false


In [86]:
for item in lookup(dpath="simulation", within=config):
    print(item)

NameError: name 'lookup' is not defined

In [100]:
import random
random.seed(42325345)
seq = ''.join([random.choice(["A", "U", "G", "C"]) for _ in range(10)])



In [102]:
seq
type(seq)

str

In [97]:
seq = []
for _ in range(10):
    seq.append(random.choice(["A", "U", "G", "C"])) 
seq
continuous_string = ''.join(seq)


'UAC'

In [110]:
masses.select(pl.col("nucleoside")).filter(pl.col("nucleoside").any not in ["U","A","G","C"])

TypeError: 'in <string>' requires string as left operand, not method

In [108]:
masses.select(pl.col("nucleoside"))

nucleoside
str
"""3483G"""
"""100G"""
"""68A"""
"""k2C"""
"""347G"""
…
"""2160A"""
"""U"""
"""chm5U"""


In [117]:
masses.filter((pl.col("nucleoside") not in ["U","A","G","C"]).all)

TypeError: the truth value of an Expr is ambiguous

You probably got here by using a Python standard library function instead of the native expressions API.
Here are some things you might want to try:
- instead of `pl.col('a') and pl.col('b')`, use `pl.col('a') & pl.col('b')`
- instead of `pl.col('a') in [y, z]`, use `pl.col('a').is_in([y, z])`
- instead of `max(pl.col('a'), pl.col('b'))`, use `pl.max_horizontal(pl.col('a'), pl.col('b'))`


In [14]:
masses.filter(~pl.col("nucleoside").is_in(["U","A","G","C"])).select(pl.col("nucleoside"))
modified_nucleoside_names = masses.filter(~pl.col("nucleoside").is_in(["U","A","G","C"])).select(pl.col("nucleoside")).to_series().to_list()
modified_nucleoside_names

['3483G',
 '100G',
 '68A',
 'k2C',
 '347G',
 'm5Um',
 '102G',
 'acp3D',
 '06A',
 'Tmoe',
 '019A',
 'm5U',
 'mcm5Um',
 '348G',
 '19A',
 '066A',
 'cmnm5se2U',
 'cmnm5ges2U',
 'm5s2U',
 'm1Y',
 'mcm5s2U',
 'm3Um',
 's2U',
 '2163A',
 'tm5U',
 'Cm',
 'cmnm5U',
 '34G',
 'mnm5s2U',
 'acp3Y',
 'm5D',
 '103G',
 'Ym',
 '2162A',
 'C+',
 '01G',
 '2165A',
 '1A',
 '1G',
 '0A',
 'inm5U',
 '69A',
 'ges2U',
 'ho5U',
 '342G',
 'nm5s2U',
 '3470G',
 'mnm5se2U',
 '62A',
 '00G',
 '3480G',
 'cm5s2U',
 'mcm5U',
 '0G',
 '2G',
 'D',
 '105G',
 '2164A',
 'tm5s2U',
 '22G',
 'f5Cm',
 'cnm5U',
 '66A',
 '65A',
 'm3Y',
 'm4Cm',
 's2Um',
 'acp3U',
 'm5Cmoe',
 '60A',
 '2A',
 '27G',
 'nm5se2U',
 '104G',
 '4G',
 'mnm5ges2U',
 'f5C',
 'hm5C',
 '2161A',
 'Uf',
 '106G',
 '61A',
 '227G',
 '34830G',
 'cmnm5Um',
 'm5Cm',
 'mchm5U',
 'm4,4Cm',
 '10G',
 'nchm5U',
 'nm5U',
 'Cf',
 'inm5Um',
 'ncm5s2U',
 '67A',
 '8A',
 'ac4C',
 '101G',
 'm4,4C',
 'Um',
 'm1acp3Y',
 '6A',
 'mcmo5U',
 '47G',
 '42G',
 '02G',
 'mchm5Um',
 's2C',
 'ac4C

In [13]:
len(nucleoside_list)

139

In [44]:
mutation_rate = 0.1
weights = [(1.-mutation_rate/len(modified_nucleoside_names))/4]*4 + [mutation_rate/len(modified_nucleoside_names)]*len(modified_nucleoside_names)


In [45]:
weights

[0.2498201438848921,
 0.2498201438848921,
 0.2498201438848921,
 0.2498201438848921,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0

In [26]:
sum(weights)

1.0009928057553956

In [ ]:
''.join([random.choices(["A", "U", "G", "C"] + modified_nucleoside_names , weights = weights ) for _ in range(10)])

TypeError: sequence item 0: expected str instance, list found

In [32]:
print(["A", "U", "G", "C"] + modified_nucleoside_names)

['A', 'U', 'G', 'C', '3483G', '100G', '68A', 'k2C', '347G', 'm5Um', '102G', 'acp3D', '06A', 'Tmoe', '019A', 'm5U', 'mcm5Um', '348G', '19A', '066A', 'cmnm5se2U', 'cmnm5ges2U', 'm5s2U', 'm1Y', 'mcm5s2U', 'm3Um', 's2U', '2163A', 'tm5U', 'Cm', 'cmnm5U', '34G', 'mnm5s2U', 'acp3Y', 'm5D', '103G', 'Ym', '2162A', 'C+', '01G', '2165A', '1A', '1G', '0A', 'inm5U', '69A', 'ges2U', 'ho5U', '342G', 'nm5s2U', '3470G', 'mnm5se2U', '62A', '00G', '3480G', 'cm5s2U', 'mcm5U', '0G', '2G', 'D', '105G', '2164A', 'tm5s2U', '22G', 'f5Cm', 'cnm5U', '66A', '65A', 'm3Y', 'm4Cm', 's2Um', 'acp3U', 'm5Cmoe', '60A', '2A', '27G', 'nm5se2U', '104G', '4G', 'mnm5ges2U', 'f5C', 'hm5C', '2161A', 'Uf', '106G', '61A', '227G', '34830G', 'cmnm5Um', 'm5Cm', 'mchm5U', 'm4,4Cm', '10G', 'nchm5U', 'nm5U', 'Cf', 'inm5Um', 'ncm5s2U', '67A', '8A', 'ac4C', '101G', 'm4,4C', 'Um', 'm1acp3Y', '6A', 'mcmo5U', '47G', '42G', '02G', 'mchm5Um', 's2C', 'ac4Cm', 'm4C', 'ncm5U', 'ho5C', '662A', '64A', 'm3U', 'cm5U', 'inm5s2U', 'mo5U', '01A', 'm3C

In [43]:
''.join(random.choices(["A", "U", "G", "C"] + modified_nucleoside_names , weights = weights, k=10))

'CUUCGGCGAU'

In [38]:
''.join([random.choices(["A", "U", "G", "C"] + modified_nucleoside_names , weights = weights ) for _ in range(10)])

TypeError: sequence item 0: expected str instance, list found

In [39]:
[random.choice(["A", "U", "G", "C"]) for _ in range(10)]

['U', 'A', 'U', 'G', 'G', 'G', 'A', 'G', 'U', 'A']

In [40]:
random.choice(["A", "U", "G", "C"])

'G'

In [46]:
ext = pl.read_csv("../results/simulation/CAGCAAAACGAUCAUAUAGAGAUCCGCAGU/30.tsv", separator="\t")

In [47]:
ext

left,right,is_start(5'),is_end(3'),single_nucleoside,true_nucleoside_mass,true_mass_with_backbone,observed_mass
i64,i64,bool,bool,bool,f64,f64,f64
0,11,true,false,false,3142.08994,3823.708086,3823.768696
12,29,false,true,false,4683.56742,5798.942568,5798.88407
0,11,true,false,false,3142.08994,3823.708086,3823.718764
12,29,false,true,false,4683.56742,5798.942568,5798.917598
0,29,true,true,false,7825.65736,9622.650654,9622.537301
…,…,…,…,…,…,…,…
0,29,true,true,false,7825.65736,9622.650654,9622.518846
0,29,true,true,false,7825.65736,9622.650654,9622.666081
0,29,true,true,false,7825.65736,9622.650654,9622.664509


In [48]:
sdf = []

In [49]:
sdf + ["asd"]

['asd']